# Лабораторная работа №2. Формирование отчётов в Apache Spark

**Задание:** Сформировать отчёт с информацией о 10 наиболее популярных языках программатирования по итогам года за период с 2010 по 2020 годы.

**Данные:**
- `posts_sample.xml` — выборка постов Stack Overflow
- `programming-languages.csv` — список языков программирования

**Результат:** набор таблиц «топ-10» для каждого года, сохранённый в формате Apache Parquet.

## 1. Инициализация SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, row_number
from pyspark.sql.window import Window
import re

spark = SparkSession.builder \
    .appName("TopLanguagesReport") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

## 2. Загрузка списка языков программирования

In [3]:
languages_df = spark.read.csv("/content/programming-languages.csv", header=True)

languages_set = set(
    row["name"].strip().lower()
    for row in languages_df.collect()
    if row["name"] is not None
)

languages_broadcast = sc.broadcast(languages_set)

## 3. Парсинг XML и извлечение пар (год, язык)

In [4]:
def parse_row(line):
    results = []
    date_match = re.search(r'CreationDate="([^"]+)"', line)
    if not date_match:
        return results
    year = int(date_match.group(1)[:4])
    if year < 2010 or year > 2020:
        return results

    tags_match = re.search(r'Tags="([^"]*)"', line)
    if not tags_match:
        return results

    tags_decoded = tags_match.group(1).replace("&lt;", "<").replace("&gt;", ">")
    tags = re.findall(r"<([^>]+)>", tags_decoded)

    langs = languages_broadcast.value
    for tag in tags:
        if tag.lower() in langs:
            results.append((year, tag.lower()))
    return results

raw_rdd = sc.textFile("/content/posts_sample.xml")
rows_rdd = raw_rdd.filter(lambda line: line.strip().startswith("<row"))

parsed_rdd = rows_rdd.flatMap(parse_row)

## 4. Подсчёт топ-10 языков по годам

In [5]:
posts_df = spark.createDataFrame(parsed_rdd, ["year", "language"])

lang_counts = posts_df.groupBy("year", "language") \
    .agg(count("*").alias("count")) \
    .orderBy("year", col("count").desc())

window_spec = Window.partitionBy("year").orderBy(col("count").desc())

top10 = lang_counts \
    .withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 10) \
    .orderBy("year", "rank")

top10.show(110, truncate=False)

+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|java       |52   |1   |
|2010|php        |46   |2   |
|2010|javascript |44   |3   |
|2010|python     |26   |4   |
|2010|objective-c|23   |5   |
|2010|c          |20   |6   |
|2010|ruby       |12   |7   |
|2010|delphi     |8    |8   |
|2010|bash       |3    |9   |
|2010|r          |3    |10  |
|2011|php        |102  |1   |
|2011|java       |93   |2   |
|2011|javascript |83   |3   |
|2011|python     |37   |4   |
|2011|objective-c|34   |5   |
|2011|c          |24   |6   |
|2011|ruby       |20   |7   |
|2011|perl       |9    |8   |
|2011|delphi     |8    |9   |
|2011|bash       |7    |10  |
|2012|php        |154  |1   |
|2012|javascript |132  |2   |
|2012|java       |124  |3   |
|2012|python     |69   |4   |
|2012|objective-c|45   |5   |
|2012|ruby       |27   |6   |
|2012|c          |27   |7   |
|2012|bash       |10   |8   |
|2012|r          |9    |9   |
|2012|lua        |6    |10  |
|2013|java

## 5. Сохранение отчёта в Parquet

In [7]:
top10.write.mode("overwrite").parquet("/content/top_languages_by_year.parquet")

## 6. Проверка сохранённых данных

In [10]:
result = spark.read.parquet("/content/top_languages_by_year.parquet")
result.orderBy("year", "rank").show(110, truncate=False)

+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|java       |52   |1   |
|2010|php        |46   |2   |
|2010|javascript |44   |3   |
|2010|python     |26   |4   |
|2010|objective-c|23   |5   |
|2010|c          |20   |6   |
|2010|ruby       |12   |7   |
|2010|delphi     |8    |8   |
|2010|bash       |3    |9   |
|2010|r          |3    |10  |
|2011|php        |102  |1   |
|2011|java       |93   |2   |
|2011|javascript |83   |3   |
|2011|python     |37   |4   |
|2011|objective-c|34   |5   |
|2011|c          |24   |6   |
|2011|ruby       |20   |7   |
|2011|perl       |9    |8   |
|2011|delphi     |8    |9   |
|2011|bash       |7    |10  |
|2012|php        |154  |1   |
|2012|javascript |132  |2   |
|2012|java       |124  |3   |
|2012|python     |69   |4   |
|2012|objective-c|45   |5   |
|2012|ruby       |27   |6   |
|2012|c          |27   |7   |
|2012|bash       |10   |8   |
|2012|r          |9    |9   |
|2012|lua        |6    |10  |
|2013|java

In [ ]:
spark.stop()